# Foundation Model Benchmark: S&P 500 Daily Directional Prediction

This notebook benchmarks modern time series foundation models against the xLSTM-TS paper's results
([arXiv:2408.12408](https://arxiv.org/abs/2408.12408)) on the same S&P 500 daily close price dataset.

**Baseline from the paper's own code (denoised data):**
- xLSTM-TS: 64-66% test accuracy (varies across runs, no fixed seed)
- TiDE: 64.86%
- TCN: 63.41%
- DeepTCN: 63.77%

**Models tested here:**
1. **Chronos-2** (Amazon, 120M) - current GIFT-Eval benchmark leader
2. **TimesFM 2.5** (Google, 200M) - strong zero-shot forecaster
3. **TiRex** (NX-AI, 35M) - xLSTM-based foundation model, NeurIPS 2025
4. **Moirai 2.0** (Salesforce, 11M) - efficient decoder-only model

All models run **zero-shot** (no training on this data) and predict the next day's close price.
Directional accuracy is computed the same way as the original paper.

## Setup

In [ ]:
# Install dependencies (run once)
# Uncomment the models you want to test

!pip install pandas numpy scikit-learn matplotlib seaborn tqdm

# Chronos-2
!pip install chronos-forecasting torch

# TimesFM 2.5
!pip install timesfm

# TiRex
!pip install tirex-forecasting

# Moirai 2.0
!pip install uni2ts einops huggingface_hub

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    mean_absolute_percentage_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

## Constants

Same dataset and splits as the original paper.

In [ ]:
# Dataset
FILE_NAME = 'sp500_daily'
STOCK = 'S&P 500'

# Date splits (same as paper)
TRAIN_END_DATE = '2021-01-01'
VAL_END_DATE = '2022-07-01'

# Context length for foundation models
CONTEXT_LENGTH = 150  # same as xLSTM-TS sequence length

# Prediction horizon
PREDICTION_LENGTH = 1  # 1-day ahead

## Load Data

In [ ]:
# Load the CSV
file_path = os.path.join('..', 'data', 'datasets', FILE_NAME + '.csv')
df = pd.read_csv(file_path, header=0, index_col='Date')
df.index = pd.to_datetime(df.index, utc=True).normalize().tz_localize(None)

# Keep only Close price
close = df[['Close']].copy()
print(f'Total rows: {len(close)}')
print(f'Date range: {close.index[0]} to {close.index[-1]}')
close.head()

In [ ]:
# Split into train / val / test (same as paper)
train_end = pd.Timestamp(TRAIN_END_DATE)
val_end = pd.Timestamp(VAL_END_DATE)

train_df = close[close.index < train_end]
val_df = close[(close.index >= train_end) & (close.index < val_end)]
test_df = close[close.index >= val_end]

print(f'Train: {len(train_df)} rows ({train_df.index[0]} to {train_df.index[-1]})')
print(f'Val:   {len(val_df)} rows ({val_df.index[0]} to {val_df.index[-1]})')
print(f'Test:  {len(test_df)} rows ({test_df.index[0]} to {test_df.index[-1]})')

## Metrics

Same metrics as the original paper: regression (MAE, MSE, RMSE, RMSSE, MAPE, MASE, R2) and directional classification (accuracy, recall, precision, F1).

In [ ]:
def naive_forecasting(actual, seasonality=1):
    return actual[:-seasonality]

def rmsse(actual, predicted, seasonality=1):
    q = mean_squared_error(actual, predicted) / mean_squared_error(
        actual[seasonality:], naive_forecasting(actual, seasonality))
    return np.sqrt(q)

def mase(actual, predicted, seasonality=1):
    return mean_absolute_error(actual, predicted) / mean_absolute_error(
        actual[seasonality:], naive_forecasting(actual, seasonality))

def calculate_regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'MSE': mean_squared_error(actual, predicted),
        'RMSE': root_mean_squared_error(actual, predicted),
        'RMSSE': rmsse(actual, predicted),
        'MAPE': mean_absolute_percentage_error(actual, predicted) * 100,
        'MASE': mase(actual, predicted),
        'R2': r2_score(actual, predicted),
    }

def calculate_directional_metrics(actual, predicted):
    """Compute directional accuracy from price series (same as paper)."""
    actual_dirs = (np.diff(actual.squeeze()) > 0).astype(int)
    pred_dirs = (np.diff(predicted.squeeze()) > 0).astype(int)
    return {
        'Test Accuracy': accuracy_score(actual_dirs, pred_dirs) * 100,
        'Recall': recall_score(actual_dirs, pred_dirs, pos_label=1) * 100,
        'Precision (Rise)': precision_score(actual_dirs, pred_dirs, pos_label=1) * 100,
        'Precision (Fall)': precision_score(actual_dirs, pred_dirs, pos_label=0) * 100,
        'F1 Score': f1_score(actual_dirs, pred_dirs, pos_label=1) * 100,
    }

def evaluate_model(actual, predicted, model_name):
    """Full evaluation: regression + directional metrics."""
    reg = calculate_regression_metrics(actual, predicted)
    dir_metrics = calculate_directional_metrics(actual, predicted)
    all_metrics = {**reg, **dir_metrics}
    print(f'\n--- {model_name} ---')
    for k, v in all_metrics.items():
        fmt = f'{v:.2f}%' if k in ('MAPE', 'Test Accuracy', 'Recall', 'Precision (Rise)', 'Precision (Fall)', 'F1 Score') else f'{v:.2f}'
        print(f'  {k}: {fmt}')
    return all_metrics

## Rolling Forecast Helper

Foundation models do zero-shot forecasting: given a context window of historical prices, predict the next value. We roll this across the test set one step at a time.

In [ ]:
# Prepare the full series for rolling prediction
# We need CONTEXT_LENGTH points before each test point
full_close = close['Close'].values

# Find the index where the test set starts
test_start_idx = len(train_df) + len(val_df)
test_actuals = full_close[test_start_idx:]
test_dates = close.index[test_start_idx:]

print(f'Test set: {len(test_actuals)} points')
print(f'Each prediction uses {CONTEXT_LENGTH} prior points as context')

---
## Model 1: Chronos-2 (Amazon, 120M)

Current benchmark leader on GIFT-Eval. Encoder-only transformer with group attention.

In [ ]:
from chronos import ChronosPipeline

chronos_model = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-base',
    device_map=DEVICE,
    torch_dtype=torch.float32,
)

chronos_preds = []
for i in tqdm(range(len(test_actuals)), desc='Chronos-2'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = torch.tensor(full_close[ctx_start:test_start_idx + i], dtype=torch.float32)
    forecast = chronos_model.predict(context.unsqueeze(0), prediction_length=PREDICTION_LENGTH)
    # forecast shape: (1, num_samples, prediction_length) - take median
    chronos_preds.append(forecast.median(dim=1).squeeze().item())

chronos_preds = np.array(chronos_preds)
chronos_metrics = evaluate_model(test_actuals, chronos_preds, 'Chronos-2')

---
## Model 2: TimesFM 2.5 (Google, 200M)

Decoder-only transformer with patch-based tokenization. Strong zero-shot performance.

In [ ]:
import timesfm

tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend='gpu' if DEVICE == 'cuda' else 'cpu',
        per_core_batch_size=1,
        horizon_len=PREDICTION_LENGTH,
        input_patch_len=32,
        output_patch_len=128,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id='google/timesfm-2.0-200m-pytorch',
    ),
)

timesfm_preds = []
for i in tqdm(range(len(test_actuals)), desc='TimesFM'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = full_close[ctx_start:test_start_idx + i].tolist()
    point_forecast, _ = tfm.forecast([context])
    timesfm_preds.append(point_forecast[0][0])

timesfm_preds = np.array(timesfm_preds)
timesfm_metrics = evaluate_model(test_actuals, timesfm_preds, 'TimesFM 2.5')

---
## Model 3: TiRex (NX-AI, 35M)

xLSTM-based foundation model. NeurIPS 2025. Beats models 10x its size.

In [ ]:
from tirex import TiRexPipeline

tirex_model = TiRexPipeline.from_pretrained(
    'NX-AI/TiRex',
    device_map=DEVICE,
)

tirex_preds = []
for i in tqdm(range(len(test_actuals)), desc='TiRex'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context = torch.tensor(full_close[ctx_start:test_start_idx + i], dtype=torch.float32)
    forecast = tirex_model.predict(context.unsqueeze(0), prediction_length=PREDICTION_LENGTH)
    tirex_preds.append(forecast.median(dim=1).squeeze().item())

tirex_preds = np.array(tirex_preds)
tirex_metrics = evaluate_model(test_actuals, tirex_preds, 'TiRex')

---
## Model 4: Moirai 2.0 (Salesforce, 11M)

Decoder-only transformer. 96% smaller than v1 with competitive accuracy.

In [ ]:
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

moirai_module = MoiraiModule.from_pretrained('Salesforce/moirai-2.0-R-small')

moirai_preds = []
for i in tqdm(range(len(test_actuals)), desc='Moirai 2.0'):
    ctx_start = test_start_idx + i - CONTEXT_LENGTH
    context_vals = full_close[ctx_start:test_start_idx + i]

    # Moirai expects a pandas DataFrame
    ctx_df = pd.DataFrame({
        'target': context_vals,
    }, index=pd.RangeIndex(len(context_vals)))

    predictor = moirai_module.create_predictor(
        prediction_length=PREDICTION_LENGTH,
        context_length=CONTEXT_LENGTH,
        num_samples=20,
    )

    # Use GluonTS-style dataset
    from gluonts.dataset.pandas import PandasDataset
    ds = PandasDataset.from_long_dataframe(ctx_df.reset_index(), target='target', item_id='index')

    forecasts = list(predictor.predict(ds))
    moirai_preds.append(np.median(forecasts[0].samples[:, 0]))

moirai_preds = np.array(moirai_preds)
moirai_metrics = evaluate_model(test_actuals, moirai_preds, 'Moirai 2.0')

---
## Results Comparison

In [ ]:
# Collect all results
all_results = {}

# Add foundation model results (only include models that ran successfully)
for name, metrics in [('Chronos-2', chronos_metrics), ('TimesFM 2.5', timesfm_metrics),
                       ('TiRex', tirex_metrics), ('Moirai 2.0', moirai_metrics)]:
    try:
        if metrics is not None:
            all_results[name] = metrics
    except NameError:
        pass  # model wasn't run

# Paper baselines (from their GitHub notebook, denoised data)
paper_baselines = {
    'xLSTM-TS (paper)': {'MAE': 55.84, 'MSE': 4600.42, 'RMSE': 67.83, 'RMSSE': 1.56, 'MAPE': 1.37, 'MASE': 1.69, 'R2': 0.94,
                         'Test Accuracy': 66.22, 'Recall': 72.11, 'Precision (Rise)': 64.93, 'Precision (Fall)': 67.88, 'F1 Score': 68.33},
    'TiDE (paper)':     {'MAE': 23.31, 'MSE': 865.17, 'RMSE': 29.41, 'RMSSE': 0.81, 'MAPE': 0.56, 'MASE': 0.82, 'R2': 0.99,
                         'Test Accuracy': 64.86, 'Recall': 67.81, 'Precision (Rise)': 66.44, 'Precision (Fall)': 62.99, 'F1 Score': 67.12},
    'TCN (paper)':      {'MAE': 30.01, 'MSE': 1333.10, 'RMSE': 36.51, 'RMSSE': 1.01, 'MAPE': 0.71, 'MASE': 1.06, 'R2': 0.98,
                         'Test Accuracy': 63.41, 'Recall': 69.86, 'Precision (Rise)': 64.15, 'Precision (Fall)': 62.39, 'F1 Score': 66.89},
}
all_results.update(paper_baselines)

results_df = pd.DataFrame(all_results).T
results_df = results_df.round(2)

# Format percentage columns for display
display_df = results_df.copy()
pct_cols = ['MAPE', 'Test Accuracy', 'Recall', 'Precision (Rise)', 'Precision (Fall)', 'F1 Score']
for col in pct_cols:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f'{x:.2f}%')

display_df

In [ ]:
# Visual comparison: Test Accuracy
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sort by test accuracy
sorted_df = results_df.sort_values('Test Accuracy', ascending=True)

# Directional accuracy
colors = ['#2196F3' if '(paper)' in name else '#FF5722' for name in sorted_df.index]
axes[0].barh(sorted_df.index, sorted_df['Test Accuracy'], color=colors)
axes[0].axvline(x=50, color='gray', linestyle='--', alpha=0.5, label='Coin flip (50%)')
axes[0].set_xlabel('Test Accuracy (%)')
axes[0].set_title('Directional Prediction Accuracy')
axes[0].legend()

# MAE
sorted_mae = results_df.sort_values('MAE', ascending=False)
colors_mae = ['#2196F3' if '(paper)' in name else '#FF5722' for name in sorted_mae.index]
axes[1].barh(sorted_mae.index, sorted_mae['MAE'], color=colors_mae)
axes[1].set_xlabel('MAE')
axes[1].set_title('Mean Absolute Error (lower is better)')

plt.suptitle(f'{STOCK} Daily Close - Foundation Models vs Paper Baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nBlue = paper baselines (trained on denoised data)')
print('Orange = foundation models (zero-shot, no training)')

---
## Notes

**Important context for interpreting results:**

- The paper baselines were **trained** on this specific dataset (with wavelet denoising). Foundation models are running **zero-shot** with no training.
- If a foundation model matches or beats ~65% accuracy zero-shot, that's remarkable — it means pre-training on diverse time series data transfers to stock prediction without any task-specific optimization.
- Foundation models that underperform zero-shot can potentially be **fine-tuned** on this data for better results.
- The paper's xLSTM-TS results are non-deterministic (no fixed seed). Their GitHub notebook shows 66.22%, not the 71.28% claimed in the paper.